# Analyze PIONIER observations of $\kappa$ Tuc A (Ertel et al. 2014)

In this notebook, we will explore the capabilities and usage of `exoboots` by
reproducing the analysis of PIONIER observations of $\kappa$ Tuc A (HD 7788A)
as done by
[Ertel et al. (2014)](https://ui.adsabs.harvard.edu/abs/2014A%26A...570A.128E/abstract).

This example is not intended as an introduction to optical long-baseline
interferometry or the phenomenon of hot exozodiacal dust and it is assumed that
this knowledge is available.

In [ ]:
import exoboots as eb
import numpy as np

First, we load the OIFITS files, in our case just one, and create what is
called by `exoboots` a `Full_data_set`.

For that we have to supply:
* the OIFITS files to be loaded `oifits_files`, either as a single
string or as a list of strings;
* the path to the data `path_to_data`, here just the current directory;
* whether to load and later analyze the visibility amplitude $|V|$ or the
squared visibility $V^2$, selected with the keyword `VISAMP` or `VIS2` (here,
we are only interested in the squared visibility as PIONIER did not measure
squared visibilities).

In [ ]:
oifits_file = "2012-07-23_SCI_HD7788_oiDataCalib.fits"
path_to_data = "./"
vis_or_vis2 = "VIS2" #  Set "VISAMP" to select the visibility amplitude.

full_data_set = eb.data_handling.Full_data_set(
    oifits_files=oifits_file,
    path_to_data=path_to_data,
    vis_or_vis2=vis_or_vis2,
)

Next, we set up a `Bootstrapper` object. This object will hold the data set,
provide visualization routines and can be supplied with a model that can be fit
to the data via bootstrapping (also called bagging).

We need to specify:
* `N_sample`: The number of bootstrapping samples to be drawn. For exploratory
analyses, a number in the few hundreds provides fast and reliable results; the
results for the fit parameters and their uncertainties might change slightly
when changing the `rng_seed` (see below). For final analysis, the number should
be chosen high enough, that the results do not change significantly for
different seeds. Typically a number of several thousands is sufficient.
* `bootstrap_selector`: This selects the way the data is grouped and
resampled. The parameter can be set to four different integers, meaning
    * `1`: Sample the data points and treat each one individually. Thus, sample
    from all available data, regardless of which baseline (pair of telescopes)
    or observation the points belong to.
    * `2`: Sample the baselines. Group all data points that belong to a certain
    baseline. Thereby, treat data from a certain baseline fully correlated.
    This is the recommended approach as this correlation typically dominates
    ([Defrère et al, 2012](https://ui.adsabs.harvard.edu/abs/2012A%26A...546L...9D/abstract);
    [Ertel et al. 2014](ui.adsabs.harvard.edu/abs/2014A%26A...570A.128E/abstract)).
    * `3`: Sample the observations. Group data points that belong to a certain
    observation. Thereby, data from a certain observation is treated as fully
    correlated.
    * `4`: This choice differs from the previous three in producing a
    spectrally dispersed analysis. All data for a distinct wavelength is
    grouped, then sampled. For distinct wavelenths, the data then stems from
    different baselines, this means sampling by baseline like for option `2`.
    Thus, this is a spectrally dispersed version of option `2`.
* `full_data_set`: A Full_data_set object containing either visibility
amplitudes or squared visibilities (see above).
* `weight_mode`: Each bootstrap step performs a leat-squares fit using
[`lmfit`](https://lmfit.github.io/lmfit-py/). The `weight_mode` parameter sets
how weights are derived for the minimzation routine. The parameter can be set
to three different keyword strings
    * `"no weights"`: There are no weights, thus each data point has the same
    weight. This is equivalent to applying a weight of `=1` to each data point.
    * `"error"`: The measurement uncertainty $\sigma$ defines the weight. For a
    data point with index $i$ and uncertainty $\sigma_i$, its weight is defined
    as $w_i = 1/\sigma_i^2$.
    * `"points per baseline"`: The weight is set as the inverse of the number
    of data points per baseline. This is motivated by the fact that for optical
    interferometry, the different spectral data points of one baseline are
    typically highly correlated. Thus, it is reasonably to assume that only the
    entirety of data points per baseline gives one independent data point. This
    has to be considered if data with different numbers of data points per
    baseline are analyzed jointly, e.g., when combining observations of
    multiple instruments or of the same instrument but with different spectral
    dispersions.
    * `"both"`: Both weights computed with option `"error"` and `"points per
    baseline"` are combined via multiplication.
* `rng_seed`: Define the integer seed for the random number generator used to
create bootstrapping samples. With a fixed seed, the computation can be exactly
reproduced.

Here, we choose `N_sample=5000` samples for stable results, select
`bootstrap_selector=2` to mimick the approach of Ertel et al. (2014), and
supply the full_data_set we have created above. The choice of `rng_seed=2026`
is arbitrary. Regarding the weights, Ertel et al. (2014) are not definitive about
whether individual uncertainties are used in the least-squares minimzation. We
will opt to do that by setting `weight_mode="error"`.

In [ ]:
bs = eb.bootstrapping.Bootstrapper(
    N_sample=5000,
    bootstrap_selector=2,
    full_data_set=full_data_set,
    weight_mode="error",
    rng_seed=2026
)

Now let's take a look at the data!

Data points with the same color belong to the same baseline, listed in the
legend. We see the same baseline appearing three times because the OIFITS file
contains three different observations.

For crowded plots, the uncertainties can be hidden by setting
`plot_data_uncertainty = False` and the legend showing the baselines IDs via
`show_baseline_legend = False`. For this demonstration, we do not want to save
the plot by setting `save=False`.

In [ ]:
bs.plot_data(save=False)

### Model setup
Now, let's set up an analytic model for analysis using `bs.setup_model()`.\
First, we will list the available models for selection.

In [ ]:
bs.list_model_keys()

Following Ertel et al. (2014), we choose the model of a limb-darkened
photosphere plus an overresolved brightness distribution (see,
di [Folco et al. 2007](https://ui.adsabs.harvard.edu/abs/2007A&A...475..243D/abstract)
or [Ertel et al. 2014](ui.adsabs.harvard.edu/abs/2014A%26A...570A.128E/abstract)
for more information), thus `limbDarkDisk_overresolved`.

Now, let's check what parameter this function uses.

In [ ]:
bs.model_funcs["limbDarkDisk_overresolved"].model_parameters

Here, the parameters are:
* `f_cse`: the flux ratio of the circumstellar environment to the star;
* `stellar_diameter`: the stellar diameter in milliarcseconds;
* `lin_limb_dark_param`: the linear limb-darkened coefficient for the stellar
photosphere.

In the following cell, we will use dictionaries to define which parameters to
vary, provide initial values, and define boundaries within which the parameters
shall be varied.

Following Ertel et al. 2014, we set the stellar diameter to 0.739
milliarcseconds and the linear limb-darkened coefficient to 0.257
([Claret et al. 1995](https://ui.adsabs.harvard.edu/abs/1995A%26AS..114..247C/abstract)),
suitable for an effective temperature of $\sim 6500$ K and a logarithmic
surface gravity of $\sim 4.0$.

We let `f_cse` vary between -0.2 and 0.2. The negative values are not physical,
but this way we can model squared visibilities larger than 1, which can arise
due to calibration issues. Parameters without supplied bounds are given the
bounds $\pm \infty$. The API allows giving bounds to fixed parameters, but the
bounds are ignored during the fit.

In [ ]:
model_key="limbDarkDisk_overresolved"

# Select which parameters to vary and which to keep fixed.
vary_param = dict(
    stellar_diameter=False,
    lin_limb_dark_param=False,
    f_cse=True,
)

# Provide initial parameter values. For the fixed parameters, these are the
# fixed values used for the fit process.
param_init_value = dict(
    stellar_diameter=0.739,
    lin_limb_dark_param=0.257,
    f_cse=0.00,
)

# Provide parameter boundaries within which the fit routine varies the
# parameters.
param_bounds = dict(
    f_cse=(-0.2, 0.2),
)

# Initialize the analytical model.
bs.setup_model(
    model_key=model_key,
    vary_param=vary_param,
    param_init_value=param_init_value,
    param_bounds=param_bounds
)

If we plot the data again, we see that a model is shown. This is the analytic
model with the initialized parameters listed above. Listing the parameters can
be turned off by setting `set_title=False`.

In [ ]:
bs.plot_data(show_baseline_legend=False, save=False)

After everything is set up, let's run the bootstrapping fit!

In [ ]:
bs.do_bootstrapping()

### Diagnostics: Bootstrapping histograms
We should check the histograms from the Bootstrapping process to see whether
they are unimodal, indicating a single solution, or multi-modal, indicating
multiple possible solutions or parameter degeneracies.

Histograms are plotted for each varied parameter. Added are a red dashed
vertical line indicating the median value as the result for that parameter.
Orange dashed lines indicate the 0.16 and 0.84 quantiles, used to compute the
parameter uncertainties. The title gives the result for the parameter and the
"all_waves" indicates that we did a broadband analysis as opposed to analyze
each wavelength individually.

The number of histogram bins can be adjusted via the `bins` argument (the
default is 20).

In this case, the histogram appears rather symmetric and unimodal, inspiring
trust in our result.

In [ ]:
bs.plot_bootstrap_histograms(bins=50, save=False)

### Displaying results
The optimized model (the "best-fit" model) can again be viewed via the plot
routine, which now shows the optimized model and the parameter results.

A weighted and reduced $\Chi^2$ along with the degrees of freedom is given as
well. However, caution is advised in interpreting these numbers as such data is
typically highly correlated while the typical interpretations of $\Chi^2$
assume uncorrelated values.

In [ ]:
bs.plot_data(show_baseline_legend=False, save=False)

The numeric values of the results can be accessed via the `bs.results`
attribute.

In [ ]:
bs.results

As uncertainties are derived from the discrete bootstrapping sample, there are
different values for the upper and lower uncertainty (indicated by the `+Delta`
and `-Delta` prefixes). Ideally, they should be very similar in value.
Averaging those, we yield for the uncertainty of the flux ratio of the
circumstellar environment $\approx 0.16\;\%$.

Thus, we find for the flux ratio of the circumstellar environment to the host
star $f_\textrm{CSE} = (1.46 \pm 0.16)\;\%$. Ertel et al. 2014 find
$f_\textrm{CSE} = (1.43 \pm 0.17)\;\%$.\
The discrepancy is not significant and can be explained by different settings
for the bootstrapping procedure such as number of steps or computation of
weights in the least-squares minimization, about which Ertel et al. 2014 are
silent about.

## Analyzing multiple observations consecutively

[Ertel et al. 2016](https://ui.adsabs.harvard.edu/abs/2016A%26A...595A..44E/abstract)
added two more observational epochs to $\kappa$ Tuc A in the years 2013 and
2014.

Here is an example for a compacted routine to analyze all three observations:

In [ ]:
oifits_files = [
    "2012-07-23_SCI_HD7788_oiDataCalib.fits",
    "2013-08-10_SCI_HD7788_oidataCalibrated.fits",
    "2014-10-11_SCI_HD7788_oidataCalibrated.fits"
]
path_to_data = "./"
vis_or_vis2 = "VIS2"

bs_multi_epochs = []

for i, oifits_file in enumerate(oifits_files):

    full_data_set = eb.data_handling.Full_data_set(
        oifits_files=oifits_file,
        path_to_data=path_to_data,
        vis_or_vis2=vis_or_vis2,
    )

    bs_multi_epochs.append(
        eb.bootstrapping.Bootstrapper(
            N_sample=5000,
            bootstrap_selector=2,
            full_data_set=full_data_set,
            weight_mode="error",
            rng_seed=2026
        )
    )

    bs_multi_epochs[i].setup_model(
        model_key=model_key,
        vary_param=vary_param,
        param_init_value=param_init_value,
        param_bounds=param_bounds
    )

    bs_multi_epochs[i].do_bootstrapping()
    bs_multi_epochs[i].plot_data(show_baseline_legend=False, save=False)

Here is an example how to read and print the numeric results.

In [ ]:
for i, oifits_file in enumerate(oifits_files):

    results = bs_multi_epochs[i].results
    f_cse = results["f_cse"]
    f_cse_unc = 0.5 * (results["+Delta f_cse"]+results["-Delta f_cse"])

    print(f"Analyzed OIFITS file: {oifits_file}")
    print(f"Exoboots analysis: f_cse = ({np.round(100*f_cse, 2)} +/- "
          f"{np.round(100*f_cse_unc, 2)})%")

    print()

## Restricting wavelengths

By default, `exoboots` uses all wavelengths present in the OIFITS file (that
are not flagged). However, it can be desired to restrict the analysis to
certain wavelength intervals.

This can be achieved by supplying the `waves` parameter to the `Full_data_set.`
that contains the desired wavelength interval in units of meters.

The `waves` argument has to be a list containing a 2-tuple with the lower
and upper interval borders as entries. The list can contain multiple tuples for
different OIFITS files or just one that will be applied to all observations.

As an example, let's load the 2012 observation and select only the center one
of the three wavelengths.  
Note that the print-out of the `oifits` module remains the same as the
wavelength selection happens afterwards.

In [ ]:
oifits_file = "2012-07-23_SCI_HD7788_oiDataCalib.fits"
path_to_data = "./"
vis_or_vis2 = "VIS2"

# Select an interval that admits only the center wavelength of the H-band
# observation.
waves = [(1.65e-6, 1.7e-6)]

full_data_set = eb.data_handling.Full_data_set(
    oifits_files=oifits_file,
    waves=waves,
    path_to_data=path_to_data,
    vis_or_vis2=vis_or_vis2,
)

bs_wave = eb.bootstrapping.Bootstrapper(
    N_sample=5000,
    bootstrap_selector=2,
    full_data_set=full_data_set,
    weight_mode="error",
    rng_seed=2026
    )

bs_wave.plot_data(show_baseline_legend=False, save=False)

## Exclude baselines

Sometimes, the measurements from a particular pair of telescopes are off or
just not usable, for instance if one of the telescopes lost the fringes during
observation or measured bad photometry.

In this case, we can exclude certain baselines by explicitly flagging them when
loading the OIFITS files using the `exclude_baselines` argument. This receives
a list of lists containing the baseline IDs to exclude for each file.

Let us exclude the baselines `A1D0`, `B2D0` with the highest spatial
frequencies. In this example, this applies to three distinct baselines, because
the OIFITS file containes three observations, and hence all three baselines
with the respective ID are removed. To gain more control, the observations
would have to be split accross three different OIFITS files.


In [ ]:
oifits_file = "2012-07-23_SCI_HD7788_oiDataCalib.fits"
path_to_data = "./"
vis_or_vis2 = "VIS2"

exclude_baselines = [["A1D0", "B2D0"]]

full_data_set = eb.data_handling.Full_data_set(
    oifits_files=oifits_file,
    path_to_data=path_to_data,
    vis_or_vis2=vis_or_vis2,
    exclude_baselines_per_file=exclude_baselines
)

bs_wave = eb.bootstrapping.Bootstrapper(
    N_sample=5000,
    bootstrap_selector=2,
    full_data_set=full_data_set,
    weight_mode="error",
    rng_seed=2026
    )

bs_wave.plot_data(save=False)